### Retriever
Retriever is an interface that returns relevant docs for a query

In [2]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

embeddings = HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")

docs = [
    Document(page_content="LangChain is a framework for LLM applications", metadata={"topic": "langchain"}),
    Document(page_content="RAG combines retrieval with generation", metadata={"topic": "rag"}),
    Document(page_content="Vector databases store embeddings", metadata={"topic": "vectors"}),
    Document(page_content="Transformers use attention mechanisms", metadata={"topic": "transformers"}),
    Document(page_content="FAISS is a similarity search library", metadata={"topic": "vectors"}),
]

vectorstore = FAISS.from_documents(docs,embeddings)
print("Vector Store created")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector Store created


### Similarity Search retriever

In [4]:
retriver = vectorstore.as_retriever(search_type="similarity",
                                    search_kwargs={"k":3})

query = "How does RAG work?"
results = retriver.invoke(query)

print(f"Query: {query}")
print("Results:")
for i,doc in enumerate(results,1):
    print(f"{i}. {doc.page_content}")
    print(f"Topic: {doc.metadata['topic']}\n")

Query: How does RAG work?
Results:
1. RAG combines retrieval with generation
Topic: rag

2. LangChain is a framework for LLM applications
Topic: langchain

3. Transformers use attention mechanisms
Topic: transformers



### MMR (Maximum Marginal Relevance)

MMR balances relevance and diversity:
- Finds the documents relevant to the query 
- Ensures results are unique 
- Reduces redundancy 

In [6]:
mmr_retriever = vectorstore.as_retriever(search_type="mmr",
                                         search_kwargs={"k":3, # final no of results
                                                        "fetch_k":5, # initial pool to select from 
                                                        "lambda_mult":0.5}) # 0=diverse,1=relevant

query = "vector store"
print(f"query:{query}")
print("Similarity Search:")

sim_results = retriver.invoke(query)
for doc in sim_results:
    print(f" - {doc.page_content}")

print("\nMMR search (more diverse):")
mmr_results = mmr_retriever.invoke(query) 
for doc in mmr_results:
    print(f"  - {doc.page_content}")

query:vector store
Similarity Search:
 - Vector databases store embeddings
 - LangChain is a framework for LLM applications
 - FAISS is a similarity search library

MMR search (more diverse):
  - Vector databases store embeddings
  - LangChain is a framework for LLM applications
  - RAG combines retrieval with generation


### Custom retriever with `@chain`

Create custom retrieval logic with `@chain`

In [7]:
from langchain_core.runnables import chain 

@chain 
def custom_retriever(query: str):
    """
    Custom retriever that:
    1. Gets initial results from vector store
    2. Filters by metadata
    3. Returns the top results
    """
    results = vectorstore.similarity_search(query,k=5)
    filtered = [doc for doc in results if doc.metadata.get("topic") == "vectors"]
    return filtered[:2]

query = "search technology"
results = custom_retriever.invoke(query)

print(f"Custom Retriever Results for: {query}\n")
for doc in results:
    print(f"  {doc.page_content}")
    print(f"  Topic: {doc.metadata['topic']}\n")

Custom Retriever Results for: search technology

  FAISS is a similarity search library
  Topic: vectors

  Vector databases store embeddings
  Topic: vectors



In [8]:
query = "LangChain framework"
results_with_scores = vectorstore.similarity_search_with_score(query, k=3)

print(f"Query: {query}\n")
print("Results with scores:")
for doc, score in results_with_scores:
    print(f"  Score: {score:.4f}")
    print(f"  Content: {doc.page_content}\n")

Query: LangChain framework

Results with scores:
  Score: 0.3769
  Content: LangChain is a framework for LLM applications

  Score: 1.6531
  Content: RAG combines retrieval with generation

  Score: 1.6559
  Content: FAISS is a similarity search library

